# 14. Regeln, Prüfen, Testen

Viele Wettbewerbsaufgaben bestehen aus **Regeln**: Wann darf ein Wort getrennt werden? Wie findet man die Grundform eines
Verbs? Ein Programm, das Regeln umsetzt, macht leicht Fehler, und die Regeln selbst haben oft Lücken. Darum gehört zum Lösen
immer auch das **Prüfen**: Stimmen die Ergebnisse? Was passiert in ungewöhnlichen Fällen? Wie gut ist das Programm?

In diesem Notebook lernst du:

- Bedingungen als kleine Funktionen zu schreiben,
- mehrere Regeln nacheinander anzuwenden, wenn **die letzte gewinnt**,
- Programme mit einer **Testtabelle** automatisch zu prüfen,
- **Sonderfälle** zu finden und eigene Testdateien anzulegen,
- ein Ergebnis auf **zwei Wegen** zu berechnen und zu vergleichen.

In der Wettbewerbs-Ecke baust du die Silbentrennung aus der Aufgabe „Silbentrennung“ (2025) und bewertest, wie gut sie ist.

## 1. Bedingungen als kleine Funktionen

Regeln sprechen von Begriffen wie „Vokal“, „Konsonant“ oder „Randfeld“. Schreib für jeden solchen Begriff eine kleine Funktion,
die `True` oder `False` zurückgibt. Dann liest sich die Regel im Programm fast wie im Aufgabentext, und du kannst jeden
Begriff einzeln testen.

#### Aufgabe 1.1

Schreibe `ist_vokal(b)` (a, e, i, o, u, ä, ö, ü, groß und klein) und `ist_konsonant(b)` (jeder andere Buchstabe). Teste mit
den Buchstaben im Gerüst.

**Erwartete Ausgabe:**
```
a True False
E True False
ü True False
k False True
S False True
y False True
```

In [ ]:
def ist_vokal(b):
    return False        # ersetze diese Zeile


def ist_konsonant(b):
    return False        # ersetze diese Zeile


for b in ['a', 'E', 'ü', 'k', 'S', 'y']:
    print(b, ist_vokal(b), ist_konsonant(b))

<details>
<summary>Tipp anzeigen</summary>

`b.lower() in 'aeiouäöü'`. Für den Konsonanten benutzt du einfach `not ist_vokal(b)`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def ist_vokal(b):
    return b.lower() in 'aeiouäöü'


def ist_konsonant(b):
    return not ist_vokal(b)


for b in ['a', 'E', 'ü', 'k', 'S', 'y']:
    print(b, ist_vokal(b), ist_konsonant(b))
```

Ist das y ein Vokal? Im Deutschen mal ja (Typ), mal nein (Yoga). Hier zählt es als Konsonant. Das ist eine kleine Festlegung
für die Dokumentation. Steht sie in **einer** Funktion, kannst du sie leicht ändern.

</details>

## 2. Regeln nacheinander: die letzte gewinnt

Manchmal widersprechen sich Regeln. Die Aufgabe legt dann fest, welche gilt. Bei der Silbentrennung heißt es: „Er wendet dann
die **letzte** davon an.“ Das setzt du so um: Du beginnst mit einem Ergebnis und gehst die Regeln **der Reihe nach** mit
einzelnen `if` durch. Jede Regel, die zutrifft, überschreibt das Ergebnis. Am Ende steht das Ergebnis der letzten passenden
Regel.

Mit `elif` wäre es umgekehrt: Dort gewinnt die **erste** passende Regel (Notebook 2). Lies also genau, was die Aufgabe will.

Ein Beispiel aus dem Kino: Der Eintritt kostet 8 Euro. Regel 1: Kinder unter 12 zahlen 5 Euro. Regel 2: Montags zahlen alle
4 Euro. Regel 3: Kinder unter 6 sind frei.

In [ ]:
def preis(alter, tag):
    p = 8
    if alter < 12:              # Regel 1
        p = 5
    if tag == 'Montag':         # Regel 2
        p = 4
    if alter < 6:               # Regel 3
        p = 0
    return p


print(preis(30, 'Freitag'), preis(10, 'Freitag'), preis(10, 'Montag'), preis(4, 'Montag'))

#### Aufgabe 2.1: Was wird ausgegeben?

Überlege zuerst, was der Code ausgibt, und schreibe es auf. Führe die Zelle erst danach aus.

In [ ]:
def preis_elif(alter, tag):
    if alter < 12:
        return 5
    elif tag == 'Montag':
        return 4
    elif alter < 6:
        return 0
    return 8


print(preis_elif(30, 'Montag'), preis_elif(10, 'Montag'), preis_elif(4, 'Freitag'))

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

**Ausgabe:**
```
4 5 5
```

Mit `elif` (und `return`) gewinnt die **erste** passende Regel. Ein 10-jähriges Kind zahlt dann am Montag 5 statt 4 Euro, und
ein 4-jähriges zahlt 5 Euro, weil die Regel für Kinder unter 6 nie erreicht wird. Dieselben Regeln ergeben also andere Preise,
je nachdem, ob die erste oder die letzte gewinnt.

</details>

## 3. Automatisch testen mit einer Testtabelle

Statt nach jeder Änderung von Hand nachzusehen, schreibst du die Testfälle in zwei Listen: die **Eingaben** und die
**erwarteten Ergebnisse**. Eine kleine Funktion `teste` probiert alle durch und meldet jeden Fehler. Nach jeder Änderung am
Programm rufst du sie wieder auf und siehst sofort, ob noch alles stimmt oder ob du etwas kaputt gemacht hast.

Als Beispiel dient die Aufgabe „Verben“ (2012): Aus einer Verbform soll die Grundform werden, zum Beispiel aus „sagst“ „sagen“.
Die Regeln: Bei einem Partizip mit „ge“ vorne fällt das „ge“ weg. Eine typische Endung wird abgeschnitten (die längste zuerst
probieren). Ein übrig gebliebenes „e“ am Ende fällt weg. Dann wird „en“ angehängt.

In [ ]:
ENDUNGEN = ['test', 'tet', 'ten', 'te', 'st', 'nd', 't', 'n']       # längste zuerst


def grundform(wort):
    wort = wort.lower()
    if wort.startswith('ge') and len(wort) > 4:
        wort = wort[2:]                         # Partizip: ge- weg
    for endung in ENDUNGEN:
        if wort.endswith(endung):
            wort = wort[:-len(endung)]
            break                               # nur eine Endung abschneiden
    if wort.endswith('e'):
        wort = wort[:-1]
    return wort + 'en'


print(grundform('sagst'), grundform('geforscht'))

#### Aufgabe 3.1

Schreibe `teste(eingaben, erwartet)`, die für jedes Paar die Grundform berechnet, bei einem Fehler eine Zeile mit Eingabe,
Ergebnis und erwartetem Ergebnis ausgibt und am Ende meldet, wie viele Tests bestanden wurden. Teste mit den fünf Pflicht-
Beispielen der Aufgabe und mit den „hinterhältigen“ Eingaben aus der Zusatzaufgabe.

**Erwartete Ausgabe:**
```
5 von 5 Tests bestanden
FEHLER: gebet -> ben statt beten
FEHLER: arbeite -> arbeien statt arbeiten
FEHLER: genehmigt -> nehmigen statt genehmigen
FEHLER: mitgehalten -> mitgehalen statt mithalten
1 von 5 Tests bestanden
```

In [ ]:
def teste(eingaben, erwartet):
    pass                # ersetze diese Zeile


pflicht = ['sagst', 'leitete', 'geforscht', 'schweigend', 'trag']
pflicht_erwartet = ['sagen', 'leiten', 'forschen', 'schweigen', 'tragen']
teste(pflicht, pflicht_erwartet)

<details>
<summary>Tipp anzeigen</summary>

Eine Schleife über die Positionen `i`, darin `ergebnis = grundform(eingaben[i])` mit `erwartet[i]` vergleichen. Einen Zähler
`bestanden` erhöhst du bei jedem richtigen Ergebnis.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def teste(eingaben, erwartet):
    bestanden = 0
    for i in range(len(eingaben)):
        ergebnis = grundform(eingaben[i])
        if ergebnis == erwartet[i]:
            bestanden = bestanden + 1
        else:
            print('FEHLER:', eingaben[i], '->', ergebnis, 'statt', erwartet[i])
    print(bestanden, 'von', len(eingaben), 'Tests bestanden')


pflicht = ['sagst', 'leitete', 'geforscht', 'schweigend', 'trag']
pflicht_erwartet = ['sagen', 'leiten', 'forschen', 'schweigen', 'tragen']
teste(pflicht, pflicht_erwartet)

schwer = ['gebet', 'arbeite', 'genehmigt', 'gehört', 'mitgehalten']
schwer_erwartet = ['beten', 'arbeiten', 'genehmigen', 'hören', 'mithalten']
teste(schwer, schwer_erwartet)
```

Die Pflicht-Beispiele klappen alle, die schweren fast keins. Das ist kein Grund zur Sorge: Die Aufgabe verlangt ausdrücklich,
auch zu zeigen, **was das Programm nicht kann**. Eine Testtabelle mit Fehlschlägen und einer Erklärung, woran es liegt (bei
„genehmigt“ ist das „ge“ kein Partizip-Zeichen, bei „arbeite“ ist das „e“ Teil des Wortstamms), ist für die Dokumentation
wertvoll.

</details>

## 4. Sonderfälle und eigene Testdateien

Die Beispieldateien der Aufgabe prüfen oft nur den „normalen“ Fall. Gute Teilnehmende testen zusätzlich **Sonderfälle**, bei
denen Programme gern abstürzen oder Unsinn ausgeben:

| Sonderfall | Beispiel |
|---|---|
| leer | Datei ohne Einträge, leeres Wort, leere Liste |
| 0 oder 1 | 0 Stimmen, Dauer 0, ein einziges Element |
| Gleichstand | zwei gleich gute Lösungen |
| Rand | erstes oder letztes Feld, Ecke eines Rasters |
| groß | sehr viele Einträge (Laufzeit, Notebook 10) |
| unmöglich | Eingaben, für die es keine Lösung gibt |

Solche Fälle schreibst du in **eigene Testdateien** im Format der Aufgabe. In der Dokumentation zeigst du, was dein Programm
damit macht. Beim Bällebad (Notebook 8 und 11) könnte das so aussehen:

In [ ]:
with open('test_leer.txt', 'w', encoding='utf-8') as f:
    print('0', file=f)
with open('test_dauer0.txt', 'w', encoding='utf-8') as f:
    print('1', file=f)
    print('9a Montag 10 10 30', file=f)
with open('test_gleichstand.txt', 'w', encoding='utf-8') as f:
    print('2', file=f)
    print('5a Dienstag 8 9 25', file=f)
    print('5b Donnerstag 11 12 25', file=f)
print('Testdateien geschrieben')

#### Aufgabe 4.1

Hier ist die Bällebad-Lösung mit dem Nutzungsplan aus Notebook 11 als Funktion `maximum_aus_datei(dateiname)`. Wende sie auf
das Beispiel und auf die drei Testdateien an. Läuft sie überall fehlerfrei? Sind die Ergebnisse sinnvoll? Ändere die
Ausgabe so, dass bei 0 Bällen `Keine Sportstunden mit Bällen` erscheint.

**Erwartete Ausgabe:**
```
daten/baellebad.txt: 60 Bälle am Montag um 14 Uhr
test_leer.txt: Keine Sportstunden mit Bällen
test_dauer0.txt: Keine Sportstunden mit Bällen
test_gleichstand.txt: 25 Bälle am Dienstag um 8 Uhr
```

In [ ]:
WOCHENTAGE = ['Montag', 'Dienstag', 'Mittwoch', 'Donnerstag', 'Freitag']


def maximum_aus_datei(dateiname):
    with open(dateiname, encoding='utf-8') as f:
        zeilen = f.read().splitlines()
    plan = []
    for t in range(5):
        plan.append([0] * 24)
    for i in range(1, int(zeilen[0]) + 1):
        teile = zeilen[i].split()
        t = WOCHENTAGE.index(teile[1])
        for stunde in range(int(teile[2]), int(teile[3])):
            plan[t][stunde] = plan[t][stunde] + int(teile[4])
    bester, bester_tag, beste_stunde = 0, 0, 0
    for t in range(5):
        for stunde in range(24):
            if plan[t][stunde] > bester:
                bester, bester_tag, beste_stunde = plan[t][stunde], t, stunde
    return bester, WOCHENTAGE[bester_tag], beste_stunde


for name in ['daten/baellebad.txt', 'test_leer.txt', 'test_dauer0.txt', 'test_gleichstand.txt']:
    print(name, maximum_aus_datei(name))

<details>
<summary>Tipp anzeigen</summary>

Die Funktion selbst musst du nicht ändern. Verteile das Ergebnis mit `bester, tag, stunde = maximum_aus_datei(name)` auf drei
Variablen und unterscheide mit `if bester == 0:`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
WOCHENTAGE = ['Montag', 'Dienstag', 'Mittwoch', 'Donnerstag', 'Freitag']


def maximum_aus_datei(dateiname):
    with open(dateiname, encoding='utf-8') as f:
        zeilen = f.read().splitlines()
    plan = []
    for t in range(5):
        plan.append([0] * 24)
    for i in range(1, int(zeilen[0]) + 1):
        teile = zeilen[i].split()
        t = WOCHENTAGE.index(teile[1])
        for stunde in range(int(teile[2]), int(teile[3])):
            plan[t][stunde] = plan[t][stunde] + int(teile[4])
    bester, bester_tag, beste_stunde = 0, 0, 0
    for t in range(5):
        for stunde in range(24):
            if plan[t][stunde] > bester:
                bester, bester_tag, beste_stunde = plan[t][stunde], t, stunde
    return bester, WOCHENTAGE[bester_tag], beste_stunde


for name in ['daten/baellebad.txt', 'test_leer.txt', 'test_dauer0.txt', 'test_gleichstand.txt']:
    bester, tag, stunde = maximum_aus_datei(name)
    if bester == 0:
        print(name + ': Keine Sportstunden mit Bällen')
    else:
        print(name + ':', bester, 'Bälle am', tag, 'um', stunde, 'Uhr')
```

Ohne die Änderung meldet das Programm für die leere Datei „0 Bälle am Montag um 0 Uhr“. Das ist Unsinn, denn am Montag um 0 Uhr
ist niemand in der Halle. Erst ein Test mit dem Sonderfall zeigt das. Beim Gleichstand nennt das Programm den Dienstag, weil
es nur bei einem **größeren** Wert wechselt. Die Aufgabe verlangt nur einen Zeitpunkt, das ist also in Ordnung, gehört aber in
die Dokumentation. Für die Abgabe gehören solche Testdateien mit in den Ordner.

</details>

## 5. Zwei Wege vergleichen

Eine sehr wirksame Prüfung: Berechne dasselbe auf **zwei verschiedenen Wegen** und vergleiche. Der eine Weg darf langsam und
einfach sein, Hauptsache, man kann ihm vertrauen. Stimmen beide Ergebnisse bei vielen Eingaben überein, ist der schnelle Weg
wahrscheinlich richtig. Stimmen sie nicht überein, hat einer der beiden einen Fehler.

Beim Bällebad ist der einfache Weg der aus Notebook 8: für jeden Tag und jede Stunde alle Klassen durchgehen.

#### Aufgabe 5.1

Schreibe `maximum_einfach(dateiname)`, die das Maximum ohne Nutzungsplan berechnet (drei Schleifen: Tage, Stunden, Klassen).
Sie muss nur die Anzahl der Bälle zurückgeben. Vergleiche für alle vier Dateien mit `maximum_aus_datei`.

**Erwartete Ausgabe:**
```
daten/baellebad.txt True
test_leer.txt True
test_dauer0.txt True
test_gleichstand.txt True
```

In [ ]:
def maximum_einfach(dateiname):
    return 0            # ersetze diese Zeile


for name in ['daten/baellebad.txt', 'test_leer.txt', 'test_dauer0.txt', 'test_gleichstand.txt']:
    print(name, maximum_einfach(name) == maximum_aus_datei(name)[0])

<details>
<summary>Tipp anzeigen</summary>

Lies die Zeilen in eine Liste `eintraege` (jede ein Ergebnis von `split()`). Dann drei Schleifen wie in Notebook 8, E.1.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def maximum_einfach(dateiname):
    with open(dateiname, encoding='utf-8') as f:
        zeilen = f.read().splitlines()
    eintraege = []
    for i in range(1, int(zeilen[0]) + 1):
        eintraege.append(zeilen[i].split())
    bester = 0
    for tag in WOCHENTAGE:
        for stunde in range(24):
            baelle = 0
            for e in eintraege:
                if e[1] == tag and int(e[2]) <= stunde < int(e[3]):
                    baelle = baelle + int(e[4])
            if baelle > bester:
                bester = baelle
    return bester


for name in ['daten/baellebad.txt', 'test_leer.txt', 'test_dauer0.txt', 'test_gleichstand.txt']:
    print(name, maximum_einfach(name) == maximum_aus_datei(name)[0])
```

</details>

## Wettbewerbs-Ecke: Silbentrennung

*Angelehnt an die Junioraufgabe „Silbentrennung“ (Jugendwettbewerb 2025).*

Lars hat vier Regeln für die Silbentrennung aufgestellt. Eine **Trennstelle** `p` liegt vor dem Buchstaben an Position `p`
(ab 0 gezählt). Bei „Va-ter“ ist das die Stelle 2. Für jede Stelle `p` von 1 bis `n - 1` gehst du die Regeln der Reihe nach
durch. **Die letzte passende Regel gewinnt.**

1. Zwischen zwei Konsonanten darf getrennt werden.
2. Der erste und der letzte Buchstabe dürfen nicht abgetrennt werden (also nicht an Stelle 1 und nicht an Stelle `n - 1`).
3. Bei drei Konsonanten hintereinander darf nur nach dem ersten getrennt werden, also nicht nach dem zweiten.
4. Nach einem Vokal darf getrennt werden, falls nicht zwei Konsonanten folgen.

Die Beschreibung der Beispieldateien (README) nennt ein Ergebnis dieser Regeln. Aus dem Satz „Sein Vater ist Bauer und erntet Mais.“ wird
`Se-i-n Va-te-r ist Ba-u-e-r und er-nte-t Ma-i-s.` Das ist dein erster Test.

#### Aufgabe E.1

Schreibe `trennstellen_lars(wort)`, die die Liste der erlaubten Trennstellen zurückgibt. Die Funktionen zum Markieren und zum
Trennen eines ganzen Textes stehen schon im Gerüst. Prüfe mit dem Satz aus der Aufgabe.

**Erwartete Ausgabe:**
```
Se-i-n Va-te-r ist Ba-u-e-r und er-nte-t Ma-i-s.
```

In [ ]:
def markiere(wort, stellen):
    ergebnis = ''
    for i in range(len(wort)):
        if i in stellen:
            ergebnis = ergebnis + '-'
        ergebnis = ergebnis + wort[i]
    return ergebnis


def trennstellen_lars(wort):
    return []           # ersetze diese Zeile


def trenne(wort):
    return markiere(wort, trennstellen_lars(wort))


def trenne_text(text):
    ergebnis = ''
    wort = ''
    for zeichen in text + ' ':
        if zeichen.isalpha():
            wort = wort + zeichen
        else:
            if wort != '':
                ergebnis = ergebnis + trenne(wort)
                wort = ''
            ergebnis = ergebnis + zeichen
    return ergebnis[:-1]


print(trenne_text('Sein Vater ist Bauer und erntet Mais.'))

<details>
<summary>Tipp anzeigen</summary>

Für jede Stelle `p` von 1 bis `n - 1`: `erlaubt = False`, dann vier einzelne `if` in der Reihenfolge der Regeln, jedes setzt
`erlaubt` neu. Die Buchstaben links und rechts der Stelle sind `wort[p - 1]` und `wort[p]`. Für Regel 3 brauchst du auch
`wort[p - 2]` (nur wenn `p >= 2`), für Regel 4 auch `wort[p + 1]` (nur wenn `p + 1 < n`).

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def markiere(wort, stellen):
    ergebnis = ''
    for i in range(len(wort)):
        if i in stellen:
            ergebnis = ergebnis + '-'
        ergebnis = ergebnis + wort[i]
    return ergebnis


def trennstellen_lars(wort):
    n = len(wort)
    stellen = []
    for p in range(1, n):
        erlaubt = False
        # Regel 1: zwischen zwei Konsonanten
        if ist_konsonant(wort[p - 1]) and ist_konsonant(wort[p]):
            erlaubt = True
        # Regel 2: nicht nach dem ersten und nicht vor dem letzten Buchstaben
        if p == 1 or p == n - 1:
            erlaubt = False
        # Regel 3: bei drei Konsonanten nicht nach dem zweiten
        if p >= 2 and ist_konsonant(wort[p - 2]) and ist_konsonant(wort[p - 1]) and ist_konsonant(wort[p]):
            erlaubt = False
        # Regel 4: nach einem Vokal, falls nicht zwei Konsonanten folgen
        if ist_vokal(wort[p - 1]) and not (p + 1 < n and ist_konsonant(wort[p]) and ist_konsonant(wort[p + 1])):
            erlaubt = True
        if erlaubt:
            stellen.append(p)
    return stellen


def trenne(wort):
    return markiere(wort, trennstellen_lars(wort))


def trenne_text(text):
    ergebnis = ''
    wort = ''
    for zeichen in text + ' ':
        if zeichen.isalpha():
            wort = wort + zeichen
        else:
            if wort != '':
                ergebnis = ergebnis + trenne(wort)
                wort = ''
            ergebnis = ergebnis + zeichen
    return ergebnis[:-1]


print(trenne_text('Sein Vater ist Bauer und erntet Mais.'))
```

Das Ergebnis stimmt Zeichen für Zeichen mit der README der Aufgabe überein. Damit ist auch geklärt, wie „die letzte Regel gewinnt“
gemeint ist: Regel 4 überstimmt Regel 2, darum „Se-i-n“ und „Ma-i-s“. Schön ist das nicht. Die Aufgabe fragt deshalb nach
**besseren** Regeln. Wie gut eine Verbesserung ist, misst man mit einer Bewertung.

</details>

Für die Bewertung brauchst du richtig getrennte Wörter (eine **Referenz**) und die Funktion `stellen_aus`, die aus einem
getrennten Wort wie `'Sau-er-kraut'` die Liste der Trennstellen macht (`[3, 5]`). Für jedes Wort vergleichst du die
Trennstellen des Programms mit denen der Referenz:

- **richtig:** Das Programm trennt, und die Referenz auch.
- **falsch:** Das Programm trennt, die Referenz nicht.
- **fehlend:** Die Referenz trennt, das Programm nicht.

#### Aufgabe E.2

Schreibe `bewerte()`, die über alle Referenzwörter die drei Zahlen zusammenzählt und ausgibt. Benutze `trenne` (nicht
`trennstellen_lars`), dann kannst du später verschiedene Versionen vergleichen.

**Erwartete Ausgabe:**
```
[3, 5]
richtig: 7  falsch: 18  fehlend: 4
```

In [ ]:
REFERENZ = """Mein Na-me ist Lars und ich es-se schreck-lich ger-ne Sau-er-kraut
Sein Va-ter Bau-er ern-tet Mais Ich ang-le Karp-fen""".split()


def stellen_aus(getrennt):
    stellen = []
    buchstaben = 0
    for zeichen in getrennt:
        if zeichen == '-':
            stellen.append(buchstaben)
        else:
            buchstaben = buchstaben + 1
    return stellen


def bewerte():
    pass                # ersetze diese Zeile


print(stellen_aus('Sau-er-kraut'))
bewerte()

<details>
<summary>Tipp anzeigen</summary>

Für jedes Referenzwort: `wort = referenz.replace('-', '')` ist das Wort ohne Striche, `programm = stellen_aus(trenne(wort))`
und `korrekt = stellen_aus(referenz)`. Dann zwei Schleifen: über `programm` (richtig oder falsch) und über `korrekt` (fehlend).

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
REFERENZ = """Mein Na-me ist Lars und ich es-se schreck-lich ger-ne Sau-er-kraut
Sein Va-ter Bau-er ern-tet Mais Ich ang-le Karp-fen""".split()


def stellen_aus(getrennt):
    stellen = []
    buchstaben = 0
    for zeichen in getrennt:
        if zeichen == '-':
            stellen.append(buchstaben)
        else:
            buchstaben = buchstaben + 1
    return stellen


def bewerte():
    richtig = 0
    falsch = 0
    fehlend = 0
    for referenz in REFERENZ:
        wort = referenz.replace('-', '')
        programm = stellen_aus(trenne(wort))
        korrekt = stellen_aus(referenz)
        for s in programm:
            if s in korrekt:
                richtig = richtig + 1
            else:
                falsch = falsch + 1
        for s in korrekt:
            if s not in programm:
                fehlend = fehlend + 1
    print('richtig:', richtig, ' falsch:', falsch, ' fehlend:', fehlend)


print(stellen_aus('Sau-er-kraut'))
bewerte()
```

</details>

#### Aufgabe E.3

**Für Schnelle: Eine Verbesserung.** Lars' Regel 4 trennt auch den ersten oder letzten Buchstaben ab („Se-i-n“). Füge als
**fünfte, letzte** Regel hinzu: An Stelle 1 und an Stelle `n - 1` wird nie getrennt. Definiere dazu `trenne` neu (der Name
bleibt gleich, so benutzt `bewerte` automatisch die neue Version) und vergleiche die Bewertung.

**Erwartete Ausgabe:**
```
richtig: 7  falsch: 10  fehlend: 4
Se-in Va-ter ist Ba-u-er und er-ntet Ma-is.
```

In [ ]:
def trenne(wort):
    return markiere(wort, trennstellen_lars(wort))      # ändere diese Funktion

bewerte()

<details>
<summary>Tipp anzeigen</summary>

Nimm die Stellen von `trennstellen_lars(wort)` und behalte nur die, die nicht 1 und nicht `len(wort) - 1` sind.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def trenne(wort):
    stellen = []
    for p in trennstellen_lars(wort):
        if p != 1 and p != len(wort) - 1:               # Regel 5: Rand nie abtrennen
            stellen.append(p)
    return markiere(wort, stellen)

bewerte()
print(trenne_text('Sein Vater ist Bauer und erntet Mais.'))
```

Die Zahl der falschen Trennstellen sinkt deutlich, ohne dass richtige verloren gehen. So arbeitest du dich in der echten
Aufgabe Schritt für Schritt zu besseren Regeln vor: Regel ausdenken, Bewertung vorher und nachher vergleichen, in der
Dokumentation festhalten. Mehr Verbesserungen (zum Beispiel für „au“, „ei“, „ch“ und „sch“) zeigt
`jwinf3/44_Silbentrennung/silbentrennung_loesung.ipynb`.

</details>

## Das hast du gelernt

| Werkzeug | So geht es |
|---|---|
| Begriff als Funktion | `def ist_vokal(b): return b.lower() in 'aeiouäöü'` |
| die letzte Regel gewinnt | Ergebnis setzen, dann einzelne `if` der Reihe nach, jedes überschreibt |
| die erste Regel gewinnt | `if` … `elif` … oder frühes `return` |
| Testtabelle | Listen `eingaben` und `erwartet`, eine Funktion meldet alle Abweichungen |
| Sonderfälle | eigene Testdateien: leer, 0, Gleichstand, Rand, groß, unmöglich |
| zwei Wege vergleichen | einfache, langsame Lösung als Kontrolle für die schnelle |
| Bewerten | richtig, falsch, fehlend gegenüber einer Referenz zählen |
| Versionen vergleichen | Funktion mit festem Namen neu definieren |

Im nächsten Notebook lernst du zwei neue Datentypen kennen: Wörterbücher und Mengen.